# temporary San Diego source-aligned benchmark

Run this notebook from the top in a fresh Kaggle GPU session with Internet enabled.

It retrains the frozen two-epoch QLoRA recipe, then structures 240 newly downloaded City of San Diego Get It Done descriptions. The city source has service categories but no full CivicStruct gold labels, so this reports strict schema validity and source-mapped service-domain agreement only. It does not change Evaluation v2 or the frozen result table.

The archive keeps hashes, source categories, strict-validation status, predicted service domains, and timing. It excludes the raw CSV, complaint text, raw model responses, and adapter weights. Keep this notebook and its result ZIP local until the saved result has been reviewed.


In [1]:
!pip -q install 'transformers==5.10.1' 'peft==0.20.0' 'trl==0.29.0' bitsandbytes accelerate datasets 'mlflow==3.15.1'

import hashlib
import json
import shutil
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

REPO_REF = '6c1c733fcbac86851d3f51ff2fdb4be50a97dd64'
RAW_BASE = f'https://raw.githubusercontent.com/goyashek/civic-grievance-structurer/{REPO_REF}'
SOURCE_URL = 'https://seshat.datasd.org/get_it_done_reports/get_it_done_requests_open_datasd.csv'
SOURCE_PAGE = 'https://data.sandiego.gov/datasets/get-it-done-reports/'
ROOT = Path('/kaggle/working/civicstruct_san_diego_temp')
OUTPUT = Path('/kaggle/working/civicstruct_san_diego_benchmark_output')
RAW_CSV = Path('/kaggle/working/get_it_done_requests_open_datasd.csv')
TRAIN_FILES = (
    'src/schema.py',
    'data/surface_variants.jsonl',
    'data/public_training_examples.jsonl',
    'data/dataset_manifest.json',
)

if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
ROOT.mkdir(parents=True, exist_ok=True)
RAW_CSV.unlink(missing_ok=True)

def download(url, destination, label):
    try:
        with urlopen(url, timeout=120) as response:
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(response.read())
    except URLError as exc:
        raise RuntimeError(f'Enable Kaggle Internet, then rerun the notebook. Could not download {label}.') from exc

for relative_path in TRAIN_FILES:
    download(f'{RAW_BASE}/{relative_path}', ROOT / relative_path, relative_path)
download(SOURCE_URL, RAW_CSV, 'the San Diego Get It Done CSV')
print({
    'project_source': RAW_BASE,
    'source_csv_bytes': RAW_CSV.stat().st_size,
    'fetched_project_files': list(TRAIN_FILES),
    'internal_test_loaded': False,
    'frozen_external_slice_loaded': False,
})


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 76.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 97.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━

In [2]:
import csv
import gc
import math
import os
import platform
import random
import re
import sys
import time
import zipfile
from collections import Counter, defaultdict
from importlib.metadata import version

sys.path.insert(0, str(ROOT))
import mlflow
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from src.schema import validate_gold

SEED = 42
MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'
MODEL_REVISION = 'a07cc9a04f16550a088caea529712d1d335b0ac1'
TRAIN_EPOCHS = 2
MAX_NEW_TOKENS = 256
MAX_LENGTH = 768
BATCH_SIZE = 4
MODEL_DTYPE = torch.float16
SAMPLE_PLAN = {
    'Street Light Maintenance': ('roads_and_streetlights', 60),
    'Sidewalk Repair Issue': ('roads_and_streetlights', 60),
    'Pavement Maintenance': ('roads_and_streetlights', 60),
    'Illegal Dumping': ('sanitation_and_waste', 60),
}

random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a Kaggle GPU runtime before running this notebook.'
torch.cuda.manual_seed_all(SEED)
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri((OUTPUT / 'mlruns').as_uri())
mlflow.set_experiment('civicstruct-san-diego-temp-benchmark')

def json_default(value):
    if isinstance(value, set):
        return sorted(value, key=str)
    if isinstance(value, (Path, os.PathLike)):
        return os.fspath(value)
    if isinstance(value, torch.dtype):
        return str(value)
    if hasattr(value, 'item'):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass
    if hasattr(value, 'tolist'):
        return value.tolist()
    return str(value)

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=json_default), encoding='utf-8')

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def text_sha256(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

print({
    'device': torch.cuda.get_device_name(0),
    'python': platform.python_version(),
    'torch': torch.__version__,
    'sample_rows': sum(quota for _, quota in SAMPLE_PLAN.values()),
})


2026/08/17 10:30:18 INFO mlflow.tracking.fluent: Experiment with name 'civicstruct-san-diego-temp-benchmark' does not exist. Creating a new experiment.


{'device': 'Tesla T4', 'python': '3.12.13', 'torch': '2.10.0+cu128', 'sample_rows': 240}


In [3]:
manifest = json.loads((ROOT / 'data/dataset_manifest.json').read_text(encoding='utf-8'))
for relative_path in ('data/surface_variants.jsonl', 'data/public_training_examples.jsonl'):
    assert sha256(ROOT / relative_path) == manifest['sha256'][relative_path], relative_path

surfaces = load_jsonl(ROOT / 'data/surface_variants.jsonl')
public_training = load_jsonl(ROOT / 'data/public_training_examples.jsonl')
controlled_training = [row for row in surfaces if row['split'] == 'train']
training = [dict(row) for row in controlled_training] + [dict(row) for row in public_training]

assert len(controlled_training) == 120
assert len(public_training) == 40
assert len(training) == 160
assert len({row['case_id'] for row in training}) == len(training)
assert all('test' not in relative_path for relative_path in TRAIN_FILES)
assert all(row['split'] == 'train' for row in training)

save_json(OUTPUT / 'frozen_recipe_source.json', {
    'project_commit': REPO_REF,
    'dataset_version': manifest['dataset_version'],
    'training_rows': len(training),
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'epochs': TRAIN_EPOCHS,
    'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
    'official_evaluation_metrics_changed': False,
})
print({
    'dataset_version': manifest['dataset_version'],
    'training_rows': len(training),
    'internal_test_loaded': False,
    'frozen_external_slice_loaded': False,
})


{'dataset_version': 'frozen_full_v2', 'training_rows': 160, 'internal_test_loaded': False, 'frozen_external_slice_loaded': False}


In [4]:
DOMAINS = ['public_transport', 'water_supply', 'sanitation_and_waste', 'roads_and_streetlights', 'electricity', 'welfare_or_document_service', 'other']
ISSUES = ['delay_or_non_arrival', 'service_outage_or_non_delivery', 'damaged_infrastructure', 'overcharging_or_payment_problem', 'record_or_document_error', 'staff_conduct', 'safety_or_health_hazard', 'other']
URGENCY = ['routine', 'time_sensitive', 'safety_critical']
MISSING = ['exact_location', 'date_or_time', 'service_identifier', 'transaction_or_reference_id', 'amount', 'supporting_evidence', 'affected_person_or_group', 'none']
SYSTEM_PROMPT = (
    'Structure one public-service complaint as exactly one JSON object. Use these fields in this order: '
    'service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, urgency, missing_information, formal_summary. '
    f'Allowed service_domain values: {DOMAINS}. Allowed issue_type values: {ISSUES}. '
    f'Allowed urgency values: {URGENCY}. Allowed missing_information values: {MISSING}. '
    'Use null for absent scalar facts. Missing information must be a non-empty ordered list. Use the none label only when no important detail is missing. '
    'Do not guess facts. The formal summary must be one neutral sentence. Return no reasoning, markdown, or commentary.'
)

def messages_for(complaint):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': complaint},
    ]

def train_records(rows):
    return [
        {
            'prompt': messages_for(row['complaint']),
            'completion': [{'role': 'assistant', 'content': json.dumps(row['gold'], ensure_ascii=False, separators=(',', ':'))}],
        }
        for row in rows
    ]

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=MODEL_DTYPE,
    bnb_4bit_use_double_quant=True,
)

def load_base():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        quantization_config=quantization_config,
        dtype=MODEL_DTYPE,
        device_map='auto',
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.config.use_cache = True
    return tokenizer, model

def encoded_batch(tokenizer, message_batch):
    kwargs = {
        'tokenize': True,
        'add_generation_prompt': True,
        'return_dict': True,
        'return_tensors': 'pt',
        'padding': True,
        'truncation': True,
        'max_length': 2048,
        'enable_thinking': False,
    }
    try:
        return tokenizer.apply_chat_template(message_batch, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking')
        return tokenizer.apply_chat_template(message_batch, **kwargs)

def generate_rows(tokenizer, model, rows):
    outputs = []
    model.eval()
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        messages = [messages_for(row['complaint']) for row in batch]
        inputs = encoded_batch(tokenizer, messages).to(next(model.parameters()).device)
        prompt_lengths = inputs['attention_mask'].sum(dim=1).tolist()
        padded_length = inputs['input_ids'].shape[1]
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        responses = tokenizer.batch_decode(generated[:, padded_length:], skip_special_tokens=True)
        outputs.extend(
            {
                'case_id': row['case_id'],
                'response': response.strip(),
                'latency_seconds': elapsed / len(batch),
                'prompt_tokens': int(prompt_tokens),
            }
            for row, response, prompt_tokens in zip(batch, responses, prompt_lengths)
        )
    return outputs

def prepare_trainable_parameters(model):
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            parameter.data = parameter.data.float()
    assert all(parameter.dtype == torch.float32 for parameter in model.parameters() if parameter.requires_grad)

def train_frozen_recipe(adapter_dir):
    tokenizer, base = load_base()
    base.config.use_cache = False
    torch.cuda.reset_peak_memory_stats()
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules='all-linear',
    )
    checkpoint_dir = Path('/kaggle/working/san_diego_temp_checkpoints')
    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)
    args = SFTConfig(
        output_dir=str(checkpoint_dir),
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type='cosine',
        logging_steps=1,
        save_strategy='no',
        report_to='none',
        fp16=False,
        bf16=False,
        max_length=MAX_LENGTH,
        completion_only_loss=True,
        gradient_checkpointing=True,
        seed=SEED,
    )
    trainer = SFTTrainer(
        model=base,
        args=args,
        train_dataset=Dataset.from_list(train_records(training)),
        processing_class=tokenizer,
        peft_config=lora_config,
    )
    prepare_trainable_parameters(trainer.model)
    trainable = sum(parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in trainer.model.parameters())
    assert 0 < trainable < total
    started = time.perf_counter()
    train_result = trainer.train()
    training_seconds = time.perf_counter() - started
    losses = [item['loss'] for item in trainer.state.log_history if 'loss' in item]
    assert losses and all(math.isfinite(loss) for loss in losses)
    adapter_dir = Path(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    metadata = {
        'recipe': 'frozen_two_epoch_qlora_reproduction_for_temporary_external_benchmark',
        'project_commit': REPO_REF,
        'model_name': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'dataset_version': manifest['dataset_version'],
        'dataset_rows': len(training),
        'epochs': TRAIN_EPOCHS,
        'configuration': args.to_dict(),
        'lora': lora_config.to_dict(),
        'losses': losses,
        'train_metrics': train_result.metrics,
        'training_seconds': training_seconds,
        'peak_gpu_memory_mb': torch.cuda.max_memory_allocated() / 1024**2,
        'device': torch.cuda.get_device_name(0),
        'trainable_parameters': trainable,
        'total_parameters': total,
        'packages': {
            name: version(name)
            for name in ('torch', 'transformers', 'peft', 'trl', 'bitsandbytes', 'datasets', 'mlflow')
        },
    }
    with mlflow.start_run(run_name='san-diego-temp-qlora-training') as run:
        metadata['mlflow_run_id'] = run.info.run_id
        mlflow.log_params({
            'model_name': MODEL_NAME,
            'model_revision': MODEL_REVISION,
            'dataset_rows': len(training),
            'epochs': TRAIN_EPOCHS,
            'lora_r': 16,
            'learning_rate': 2e-4,
            'experiment_type': 'temporary_source_aligned_external_benchmark',
        })
        mlflow.log_metrics({
            'train_loss': train_result.metrics['train_loss'],
            'training_seconds': training_seconds,
            'peak_gpu_memory_mb': metadata['peak_gpu_memory_mb'],
        })
        mlflow.log_text(json.dumps(metadata, indent=2, default=json_default), 'training_metadata.json')
    return {'trainer': trainer, 'tokenizer': tokenizer, 'metadata': metadata, 'checkpoint_dir': checkpoint_dir}


In [5]:
EMAIL_RE = re.compile(r'\b[^\s@]+@[^\s@]+\.[^\s@]+\b')
URL_RE = re.compile(r'\b(?:https?://|www\.)', re.IGNORECASE)
PHONE_RE = re.compile(r'\b(?:\+?\d[\s().-]*){10,}\b')
CONTACT_CUE_RE = re.compile(r'\b(?:my name is|call me|contact me|reach me)\b', re.IGNORECASE)

def clean_text(value):
    return ' '.join(str(value or '').split()).strip()

def source_candidate(row):
    service_name = clean_text(row.get('service_name'))
    if service_name not in SAMPLE_PLAN:
        return None
    description = clean_text(row.get('public_description'))
    street_address = clean_text(row.get('street_address'))
    if street_address:
        description = re.sub(re.escape(street_address), ' ', description, flags=re.IGNORECASE)
    description = clean_text(description)
    if not 20 <= len(description) <= 600:
        return None
    if EMAIL_RE.search(description) or URL_RE.search(description) or PHONE_RE.search(description):
        return None
    if CONTACT_CUE_RE.search(description) or any(character.isdigit() for character in description):
        return None
    community = clean_text(row.get('comm_plan_name'))
    complaint = f'In {community.title()}, {description[:1].lower() + description[1:]}' if community else f'In San Diego, {description[:1].lower() + description[1:]}'
    source_hash = text_sha256(json.dumps({
        'service_name': service_name,
        'service_name_detail': clean_text(row.get('service_name_detail')),
        'community': community,
        'description': description,
    }, sort_keys=True, ensure_ascii=False))
    target_domain, _ = SAMPLE_PLAN[service_name]
    return {
        'case_id': f'sd-source-{source_hash[:16]}',
        'complaint': complaint,
        'source_row_sha256': source_hash,
        'source_service_name': service_name,
        'source_service_name_detail': clean_text(row.get('service_name_detail')),
        'target_domain': target_domain,
    }

source_file_sha256 = sha256(RAW_CSV)
source_file_bytes = RAW_CSV.stat().st_size
candidates = defaultdict(dict)
with RAW_CSV.open(encoding='utf-8-sig', newline='') as handle:
    reader = csv.DictReader(handle)
    required_columns = {'service_name', 'service_name_detail', 'comm_plan_name', 'street_address', 'public_description'}
    missing_columns = required_columns - set(reader.fieldnames or ())
    if missing_columns:
        raise RuntimeError(f'San Diego CSV is missing required columns: {sorted(missing_columns)}')
    source_columns = reader.fieldnames
    for row in reader:
        candidate = source_candidate(row)
        if candidate is not None:
            candidates[candidate['source_service_name']][candidate['source_row_sha256']] = candidate

benchmark_rows = []
available_counts = {}
for service_name, (_, quota) in SAMPLE_PLAN.items():
    selected = sorted(candidates[service_name].values(), key=lambda row: row['source_row_sha256'])[:quota]
    available_counts[service_name] = len(candidates[service_name])
    if len(selected) != quota:
        raise RuntimeError(
            f'Only {len(selected)} privacy-filtered rows were available for {service_name}; expected {quota}. '
            f'Available counts: {available_counts}'
        )
    benchmark_rows.extend(selected)

assert len(benchmark_rows) == sum(quota for _, quota in SAMPLE_PLAN.values())
assert len({row['source_row_sha256'] for row in benchmark_rows}) == len(benchmark_rows)
assert Counter(row['source_service_name'] for row in benchmark_rows) == Counter({
    service_name: quota for service_name, (_, quota) in SAMPLE_PLAN.items()
})
del candidates
RAW_CSV.unlink()
assert not RAW_CSV.exists()

source_metadata = {
    'source_dataset': 'City of San Diego Get It Done service requests',
    'source_page': SOURCE_PAGE,
    'source_csv': SOURCE_URL,
    'source_file_sha256': source_file_sha256,
    'source_file_bytes': source_file_bytes,
    'source_columns_seen': source_columns,
    'sample_plan': {
        service_name: {'target_domain': target_domain, 'quota': quota, 'privacy_filtered_available': available_counts[service_name]}
        for service_name, (target_domain, quota) in SAMPLE_PLAN.items()
    },
    'rows_retained': len(benchmark_rows),
    'privacy_filter': (
        'removed exact source street-address text; then rejected descriptions with URLs, emails, phone-like strings, '
        'contact cues, digits, or lengths outside 20 to 600 characters. Raw CSV and complaint text are not archived.'
    ),
    'full_schema_gold_available': False,
    'official_evaluation_metrics_changed': False,
}
save_json(OUTPUT / 'san_diego_source_metadata.json', source_metadata)
print({
    'benchmark_rows': len(benchmark_rows),
    'per_category': dict(Counter(row['source_service_name'] for row in benchmark_rows)),
    'raw_csv_removed': not RAW_CSV.exists(),
    'full_schema_gold_available': False,
})


{'benchmark_rows': 240, 'per_category': {'Street Light Maintenance': 60, 'Sidewalk Repair Issue': 60, 'Pavement Maintenance': 60, 'Illegal Dumping': 60}, 'raw_csv_removed': True, 'full_schema_gold_available': False}


In [6]:
adapter_dir = OUTPUT / 'temporary_adapter'
training_result = train_frozen_recipe(adapter_dir)
save_json(OUTPUT / 'training_metadata.json', training_result['metadata'])

del training_result['trainer'], training_result['tokenizer']
gc.collect()
torch.cuda.empty_cache()

reload_tokenizer, reload_base = load_base()
model = PeftModel.from_pretrained(reload_base, adapter_dir)
raw_outputs = generate_rows(reload_tokenizer, model, benchmark_rows)

def parse_strict(response):
    try:
        parsed = json.loads(response)
        validate_gold(parsed)
        return parsed
    except (json.JSONDecodeError, TypeError, ValueError):
        return None

records = []
for row, output in zip(benchmark_rows, raw_outputs):
    parsed = parse_strict(output['response'])
    records.append({
        'source_row_sha256': row['source_row_sha256'],
        'source_service_name': row['source_service_name'],
        'source_service_name_detail': row['source_service_name_detail'],
        'target_service_domain': row['target_domain'],
        'predicted_service_domain': None if parsed is None else parsed['service_domain'],
        'strict_schema_valid': parsed is not None,
        'raw_response_sha256': text_sha256(output['response']),
        'latency_seconds': output['latency_seconds'],
        'prompt_tokens': output['prompt_tokens'],
    })

valid_records = [record for record in records if record['strict_schema_valid']]
end_to_end_matches = [
    record['strict_schema_valid'] and record['predicted_service_domain'] == record['target_service_domain']
    for record in records
]
valid_only_matches = [
    record['predicted_service_domain'] == record['target_service_domain']
    for record in valid_records
]
benchmark_summary = {
    'experiment_type': 'temporary_source_aligned_external_benchmark',
    'official_evaluation_metrics_changed': False,
    'full_schema_gold_available': False,
    'comparison_note': 'This is not a replacement for Evaluation v2. The city service category is only used as a mapped service-domain reference.',
    'model': {'name': MODEL_NAME, 'revision': MODEL_REVISION, 'adapter_recipe': 'frozen_two_epoch_qlora_reproduction'},
    'dataset': {
        'source': 'City of San Diego Get It Done service requests',
        'source_file_sha256': source_file_sha256,
        'rows': len(records),
        'category_counts': dict(Counter(record['source_service_name'] for record in records)),
        'target_domain_counts': dict(Counter(record['target_service_domain'] for record in records)),
    },
    'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS, 'batch_size': BATCH_SIZE},
    'metrics': {
        'strict_schema_validity_rate': len(valid_records) / len(records),
        'strict_schema_valid_count': len(valid_records),
        'source_domain_match_rate_end_to_end': sum(end_to_end_matches) / len(records),
        'source_domain_match_rate_valid_only': None if not valid_records else sum(valid_only_matches) / len(valid_records),
        'mean_latency_seconds': sum(record['latency_seconds'] for record in records) / len(records),
        'mean_prompt_tokens': sum(record['prompt_tokens'] for record in records) / len(records),
    },
    'source_domain_confusion': dict(Counter(
        f"{record['target_service_domain']}->{record['predicted_service_domain'] or 'invalid'}"
        for record in records
    )),
    'training_run_id': training_result['metadata']['mlflow_run_id'],
    'environment': {
        'device': torch.cuda.get_device_name(0),
        'packages': {
            name: version(name)
            for name in ('torch', 'transformers', 'peft', 'trl', 'bitsandbytes', 'datasets', 'mlflow')
        },
    },
}
assert len(records) == 240
assert benchmark_summary['metrics']['strict_schema_valid_count'] <= 240
save_json(OUTPUT / 'san_diego_benchmark_summary.json', benchmark_summary)
save_json(OUTPUT / 'san_diego_benchmark_records.json', records)

with mlflow.start_run(run_name='san-diego-temp-source-aligned') as run:
    benchmark_summary['benchmark_run_id'] = run.info.run_id
    mlflow.log_params({
        'experiment_type': benchmark_summary['experiment_type'],
        'model_name': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'rows': len(records),
        'source_file_sha256': source_file_sha256,
        'full_schema_gold_available': False,
        'official_evaluation_metrics_changed': False,
    })
    mlflow.log_metrics({key: value for key, value in benchmark_summary['metrics'].items() if value is not None})
    mlflow.log_artifact(str(OUTPUT / 'san_diego_benchmark_summary.json'))
    mlflow.log_artifact(str(OUTPUT / 'san_diego_benchmark_records.json'))
save_json(OUTPUT / 'san_diego_benchmark_summary.json', benchmark_summary)

del raw_outputs, model, reload_base, reload_tokenizer, benchmark_rows
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(adapter_dir)
shutil.rmtree(training_result['checkpoint_dir'])
assert not adapter_dir.exists()
assert not training_result['checkpoint_dir'].exists()

archive = shutil.make_archive(
    '/kaggle/working/civicstruct_san_diego_external_benchmark',
    'zip',
    root_dir=str(OUTPUT),
)
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print({
    'archive': archive,
    'archive_bytes': Path(archive).stat().st_size,
    'rows': len(records),
    'strict_schema_validity_rate': benchmark_summary['metrics']['strict_schema_validity_rate'],
    'source_domain_match_rate_end_to_end': benchmark_summary['metrics']['source_domain_match_rate_end_to_end'],
    'official_evaluation_metrics_changed': False,
})


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
1,0.722799
2,0.701682
3,0.332850
4,0.247250
5,0.207690
6,0.199060
7,0.199585
8,0.180248
9,0.145300
10,0.163221


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'archive': '/kaggle/working/civicstruct_san_diego_external_benchmark.zip', 'archive_bytes': 68009, 'rows': 240, 'strict_schema_validity_rate': 0.95, 'source_domain_match_rate_end_to_end': 0.8041666666666667, 'official_evaluation_metrics_changed': False}


## Review before keeping

This is a supplemental, source-aligned check. Do not add it to the frozen Evaluation v2 table.

Before deciding whether to retain it, inspect the saved summary and records for all 240 rows, confirm the archive passes its integrity check, and keep the source category mapping and privacy filter in the result note. The archive does not contain the raw CSV, complaint text, raw model responses, or adapter.
